# Pipeline PROF2 - Small Language Model

Notebook de referência para a entrega final. Ele centraliza os comandos usados no projeto: pré-treino, continuação opcional até 1B/2B tokens, mid-training, SFT, avaliação, geração e demo.

Arquivos pesados (`data/*.bin` e `checkpoints/*.pt`) ficam locais e/ou hospedados externamente. O GitHub deve conter código, configs, logs, gráficos, JSONs de avaliação, slides, README e este notebook.

## 1. Ambiente

Use um kernel Python com PyTorch/CUDA. A célula abaixo deve mostrar `cuda disponível: True` para treinar na GPU.

In [2]:
import sys
import torch

PYTHON = sys.executable

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
print('gpus:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

python: C:\Users\celso\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
torch: 2.11.0+cu128
cuda disponível: True
gpus: 1
gpu: NVIDIA GeForce RTX 3060


## 2. Dados locais

Confere os arquivos binários usados no pré-treino e nas etapas conversacionais.

In [3]:
from pathlib import Path
import numpy as np

files = [
    ('pretrain train', Path('data/fineweb_edu_train_500m.bin'), np.uint16),
    ('pretrain val', Path('data/fineweb_edu_val_500m.bin'), np.uint16),
    ('pretrain 2b train', Path('data/fineweb_edu_train_2b.bin'), np.uint16),
    ('pretrain 2b val', Path('data/fineweb_edu_val_2b.bin'), np.uint16),
    ('mid train', Path('data/smoltalk_mid_train.bin'), np.uint16),
    ('mid val', Path('data/smoltalk_mid_val.bin'), np.uint16),
    ('sft train tokens', Path('data/smoltalk_sft_train_tokens.bin'), np.uint16),
    ('sft train labels', Path('data/smoltalk_sft_train_labels.bin'), np.int32),
    ('sft val tokens', Path('data/smoltalk_sft_val_tokens.bin'), np.uint16),
    ('sft val labels', Path('data/smoltalk_sft_val_labels.bin'), np.int32),
]

for label, path, dtype in files:
    if not path.exists():
        print(label, 'NÃO ENCONTRADO:', path)
        continue
    arr = np.memmap(path, dtype=dtype, mode='r')
    mb = path.stat().st_size / 1024 / 1024
    print(f'{label}: {len(arr):,} itens | {mb:.2f} MB | {path}')

pretrain train: 512,824,558 itens | 978.14 MB | data\fineweb_edu_train_500m.bin
pretrain val: 5,133,236 itens | 9.79 MB | data\fineweb_edu_val_500m.bin
pretrain 2b train: 2,045,505,196 itens | 3901.49 MB | data\fineweb_edu_train_2b.bin
pretrain 2b val: 20,406,594 itens | 38.92 MB | data\fineweb_edu_val_2b.bin
mid train: 45,438,471 itens | 86.67 MB | data\smoltalk_mid_train.bin
mid val: 942,713 itens | 1.80 MB | data\smoltalk_mid_val.bin
sft train tokens: 45,389,283 itens | 86.57 MB | data\smoltalk_sft_train_tokens.bin
sft train labels: 45,389,283 itens | 173.15 MB | data\smoltalk_sft_train_labels.bin
sft val tokens: 941,731 itens | 1.80 MB | data\smoltalk_sft_val_tokens.bin
sft val labels: 941,731 itens | 3.59 MB | data\smoltalk_sft_val_labels.bin


## 3. Pré-treino original - 60k steps / 491,5M tokens

Esta foi a base da Entrega 1. Rode apenas se precisar refazer ou continuar do checkpoint atual.

In [ ]:
# Recria os dados de pré-treino, se necessário.
# !"{PYTHON}" scripts/prepare_data.py --config configs/pretrain_llama_100m.yaml --max-documents 500000

In [ ]:
# Continua/roda o pré-treino local até 60k steps.
# !"{PYTHON}" src/train.py --config configs/pretrain_llama_100m_rtx3060_60k.yaml --resume checkpoints/ckpt_last.pt

## 4. Continuação opcional do pré-treino para 1B e 2B tokens

Mantém o mesmo consumo de VRAM: `batch_size=1`, `block_size=512`, `gradient_accumulation_steps=16`, ou 8.192 tokens por step.

Como já temos `fineweb_edu_train_500m.bin`, o correto é preparar somente a fatia adicional de aproximadamente 1,5B tokens, combiná-la com os 500M existentes e gerar `fineweb_edu_train_2b.bin`.

- Preparação extra: pula os primeiros 500.000 documentos já usados e processa mais 1.500.000 documentos.
- 1B: continua de `checkpoints/ckpt_last.pt` até 122.100 steps.
- 2B: continua de `checkpoints/pretrain_1b/ckpt_last.pt` até 244.200 steps.

Depois de melhorar o pré-treino, o correto é refazer mid-training e SFT a partir do novo checkpoint.

In [ ]:
# Prepara a fatia nova de ~1,5B tokens e concatena com os 500M existentes.
# Rode apenas se data/fineweb_edu_train_2b.bin e data/fineweb_edu_val_2b.bin ainda não existirem.
# !"{PYTHON}" scripts/prepare_data.py --config configs/pretrain_fineweb_extra_1_5b.yaml --skip-documents 500000 --max-documents 1500000
# !"{PYTHON}" scripts/merge_token_bins.py --output data/fineweb_edu_train_2b.bin --inputs data/fineweb_edu_train_500m.bin data/fineweb_edu_train_extra_1_5b.bin
# !"{PYTHON}" scripts/merge_token_bins.py --output data/fineweb_edu_val_2b.bin --inputs data/fineweb_edu_val_500m.bin data/fineweb_edu_val_extra_1_5b.bin

In [1]:
# Continua até aproximadamente 1B tokens vistos.
# Esta versão roda dentro do próprio kernel do notebook, sem chamar outro `python`.
from pathlib import Path
import os
import sys
import torch

ROOT = Path(r'E:\trabalhos\Prof2')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from train import run_training
from utils import load_config

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA não está disponível neste kernel. Troque o kernel do VS Code antes de treinar.')
print('gpu:', torch.cuda.get_device_name(0))

config = load_config('configs/pretrain_llama_100m_rtx3060_1b.yaml')
run_training(config, resume_path='checkpoints/ckpt_last.pt')

python: C:\Users\celso\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
torch: 2.11.0+cu128
cuda disponível: True
gpu: NVIDIA GeForce RTX 3060


E:\trabalhos\Prof2\src\train.py:143: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and dtype_name == "float16"))


resuming from checkpoints/ckpt_last.pt at step 60001
device: cuda
parameters: 109,392,384
train tokens: 2,045,505,196 | val tokens: 20,406,594
step 060020 | train 4.0856 | val 4.2957 | lr 5e-06
step 060040 | train 4.5154 | val 4.2957 | lr 5e-06
step 060060 | train 4.4363 | val 4.2957 | lr 5e-06
step 060080 | train 4.2038 | val 4.2957 | lr 5e-06
step 060100 | train 4.2540 | val 4.2957 | lr 5e-06
step 060120 | train 4.4591 | val 4.2957 | lr 5e-06
step 060140 | train 4.3314 | val 4.2957 | lr 5e-06
step 060160 | train 4.2826 | val 4.2957 | lr 5e-06
step 060180 | train 4.1970 | val 4.2957 | lr 5e-06
step 060200 | train 4.3314 | val 4.2957 | lr 5e-06
step 060220 | train 4.2458 | val 4.2957 | lr 5e-06
step 060240 | train 4.2467 | val 4.2957 | lr 5e-06
step 060260 | train 4.3040 | val 4.2957 | lr 5e-06
step 060280 | train 4.4886 | val 4.2957 | lr 5e-06
step 060300 | train 4.4037 | val 4.2957 | lr 5e-06
step 060320 | train 4.2380 | val 4.2957 | lr 5e-06
step 060340 | train 4.4506 | val 4.2957 |

In [1]:
# Continua de 1B até aproximadamente 2B tokens vistos.
# Esta célula também roda dentro do próprio kernel do notebook.
from pathlib import Path
import os
import sys
import torch

ROOT = Path(r'E:\trabalhos\Prof2')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from train import run_training
from utils import load_config

resume_path = Path('checkpoints/pretrain_1b/ckpt_last.pt')
if not resume_path.exists():
    raise FileNotFoundError(resume_path)

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA não está disponível neste kernel. Troque o kernel do VS Code antes de treinar.')
print('gpu:', torch.cuda.get_device_name(0))

config = load_config('configs/pretrain_llama_100m_rtx3060_2b.yaml')
run_training(config, resume_path=str(resume_path))

python: C:\Users\celso\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
torch: 2.11.0+cu128
cuda disponível: True
gpu: NVIDIA GeForce RTX 3060


E:\trabalhos\Prof2\src\train.py:143: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and dtype_name == "float16"))


resuming from checkpoints\pretrain_1b\ckpt_last.pt at step 122101
device: cuda
parameters: 109,392,384
train tokens: 2,045,505,196 | val tokens: 20,406,594
step 122120 | train 3.9436 | val 4.2588 | lr 5e-06
step 122140 | train 4.3657 | val 4.2588 | lr 5e-06
step 122160 | train 4.2862 | val 4.2588 | lr 5e-06
step 122180 | train 4.0692 | val 4.2588 | lr 5e-06
step 122200 | train 4.1141 | val 4.2588 | lr 5e-06
step 122220 | train 4.3127 | val 4.2588 | lr 5e-06
step 122240 | train 4.1919 | val 4.2588 | lr 5e-06
step 122260 | train 4.1364 | val 4.2588 | lr 5e-06
step 122280 | train 4.0572 | val 4.2588 | lr 5e-06
step 122300 | train 4.1902 | val 4.2588 | lr 5e-06
step 122320 | train 4.1022 | val 4.2588 | lr 5e-06
step 122340 | train 4.1115 | val 4.2588 | lr 5e-06
step 122360 | train 4.1746 | val 4.2588 | lr 5e-06
step 122380 | train 4.3356 | val 4.2588 | lr 5e-06
step 122400 | train 4.2790 | val 4.2588 | lr 5e-06
step 122420 | train 4.0965 | val 4.2588 | lr 5e-06
step 122440 | train 4.3149 |

KeyboardInterrupt: 

## 5. Mid-training e SFT

Fluxo usado na entrega final com SmolTalk. Se você treinar um novo pré-treino de 1B/2B, troque o `--init-checkpoint` do mid-training para o novo melhor checkpoint.

In [ ]:
# Preparação dos dados conversacionais.
# !"{PYTHON}" scripts/prepare_chat_data.py --config configs/midtrain_smoltalk.yaml --phase mid --max-examples 50000
# !"{PYTHON}" scripts/prepare_chat_data.py --config configs/sft_smoltalk.yaml --phase sft --max-examples 50000

In [ ]:
# Mid-training a partir do pré-treino.
# !"{PYTHON}" src/finetune.py --config configs/midtrain_smoltalk.yaml --init-checkpoint checkpoints/ckpt_best.pt

# SFT a partir do melhor checkpoint de mid-training.
# !"{PYTHON}" src/finetune.py --config configs/sft_smoltalk.yaml --init-checkpoint checkpoints/midtrain_best.pt

## 6. Avaliação e curvas

Gera curvas, perplexidade e benchmark de múltipla escolha para os artefatos da entrega.

In [ ]:
!"{PYTHON}" src/plot_loss.py --log outputs/train_log_0_60k.csv --output outputs/loss_curve_0_60k.png
!"{PYTHON}" src/plot_loss.py --log outputs/midtrain_log.csv --output outputs/midtrain_loss_curve.png
!"{PYTHON}" src/plot_loss.py --log outputs/sft_log.csv --output outputs/sft_loss_curve.png

In [ ]:
!"{PYTHON}" src/evaluate.py val_curve --log outputs/train_log_0_60k.csv --output outputs/pretrain_val_curve.json
!"{PYTHON}" src/evaluate.py perplexity --checkpoint checkpoints/sft_best.pt --data data/fineweb_edu_val_500m.bin --eval-iters 100 --batch-size 1 --output outputs/sft_perplexity.json
!"{PYTHON}" src/evaluate.py multiple_choice --checkpoint checkpoints/sft_best.pt --benchmark hellaswag --split validation --max-examples 200 --output outputs/sft_hellaswag_200.json

## 7. Geração e demo

Use geração simples para sanity check e Streamlit para a demo final.

In [ ]:
!"{PYTHON}" src/generate.py --checkpoint checkpoints/sft_best.pt --prompt 'User:\nWhat is the importance of study?\nAssistant:\n' --max_new_tokens 100 --temperature 0.7 --top_k 50 --output outputs/samples_sft.txt

In [ ]:
# Rode no terminal do VS Code, não dentro do notebook:
# python -m streamlit run app.py

## 8. O que salvar no GitHub e o que não salvar

Salvar no GitHub:

- código em `src/`, `scripts/` e `app.py`;
- configs em `configs/`;
- `README.md`, `Treino.ipynb`, `requirements.txt`;
- logs, curvas e JSONs em `outputs/`;
- slides em `slides/`;
- READMEs auxiliares em `data/` e `checkpoints/`.

Não salvar no GitHub:

- `data/*.bin`;
- `checkpoints/*.pt` e `checkpoints/**/*.pt`;
- caches `__pycache__/`;
- `.vscode/`;
- PDF do enunciado em `pdf/`.

Hospedar fora do GitHub:

- `checkpoints/sft_best.pt` para a entrega final;
- se treinar 1B/2B, também hospedar o melhor checkpoint novo.